In [ ]:
import os

DATA_DIR = os.path.join(os.path.dirname(os.path.abspath(__file__)), "data")

def load_documents(data_dir: str = DATA_DIR):
    
    documents = []
    for filename in sorted(os.listdir(data_dir)):
        if filename.endswith(".txt"):
            path = os.path.join(data_dir, filename)
            with open(path, "r", encoding="utf-8") as f:
                text = f.read()
            documents.append({"source": filename, "text": text})
    return documents

if __name__ == "__main__":
    docs = load_documents()
    print(f"تم تحميل {len(docs)} مستند:")
    for d in docs:
        print(f" - {d['source']} ({len(d['text'])} حرف)")


In [ ]:
import re
import os
import importlib.util

def _load_module(filename, modname):
    path = os.path.join(os.path.dirname(os.path.abspath(__file__)), filename)
    spec = importlib.util.spec_from_file_location(modname, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

def preprocess_text(raw_text: str) -> str:
    
    text = raw_text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def preprocess_documents(documents):
    
    cleaned = []
    for doc in documents:
        cleaned.append({
            : doc["source"],
            : preprocess_text(doc["text"]),
        })
    return cleaned

if __name__ == "__main__":
    documents_mod = _load_module("01_documents.py", "documents_mod")
    docs = documents_mod.load_documents()
    cleaned_docs = preprocess_documents(docs)
    for d in cleaned_docs:
        print(f"{d['source']}: بعد التنظيف {len(d['text'])} حرف")


In [ ]:
import re
import os
import importlib.util

def _load_module(filename, modname):
    path = os.path.join(os.path.dirname(os.path.abspath(__file__)), filename)
    spec = importlib.util.spec_from_file_location(modname, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

ARTICLE_PATTERN = re.compile(
    
    ,
    re.UNICODE,
)

def chunk_text(text: str, source: str):
    parts = ARTICLE_PATTERN.split(text)
    chunks = []
    current_header = None
    for part in parts:
        part = part.strip()
        if not part:
            continue
        if ARTICLE_PATTERN.fullmatch(part):
            current_header = part
        else:
            if current_header:
                num_match = re.search(r"\d+", current_header)
                article_num = num_match.group() if num_match else str(len(chunks) + 1)
                clean_source = source.replace(".txt", "")
                chunks.append({
                    : f"{clean_source}_art{article_num}_{len(chunks)}",
                    : source,
                    : article_num,
                    : f"{current_header} {part}".strip(),
                })
                current_header = None
            else:
                if len(part) > 30:
                    clean_source = source.replace(".txt", "")
                    chunks.append({
                        : f"{clean_source}_preamble_{len(chunks)}",
                        : source,
                        : "0",
                        : part,
                    })
    return chunks

def chunk_documents(documents):
    
    all_chunks = []
    for doc in documents:
        all_chunks.extend(chunk_text(doc["text"], doc["source"]))
    return all_chunks

if __name__ == "__main__":
    documents_mod = _load_module("01_documents.py", "documents_mod")
    preprocessing_mod = _load_module("02_preprocessing.py", "preprocessing_mod")

    docs = documents_mod.load_documents()
    cleaned_docs = preprocessing_mod.preprocess_documents(docs)
    chunks = chunk_documents(cleaned_docs)

    print(f"تم استخراج {len(chunks)} مادة/جزء")
    for c in chunks[:5]:
        print(f"- {c['id']}: {c['text'][:70]}...")


In [ ]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

_model = None

def get_model():
    global _model
    if _model is None:
        _model = SentenceTransformer(MODEL_NAME)
    return _model

def embed_texts(texts):
    
    model = get_model()
    embeddings = model.encode(texts, normalize_embeddings=True, show_progress_bar=False)
    return embeddings.tolist()

if __name__ == "__main__":
    sample = ["هل يحق للزوجة طلب الخلع؟", "نفقة الأولاد على الأب"]
    vectors = embed_texts(sample)
    print(f"تم عمل embeddings لـ {len(sample)} جملة، أبعاد كل متجه: {len(vectors[0])}")


In [ ]:
import os
import importlib.util
import chromadb

CHROMA_DIR = os.path.join(os.path.dirname(os.path.abspath(__file__)), "chroma_db")
COLLECTION_NAME = "family_law"

def _load_module(filename, modname):
    path = os.path.join(os.path.dirname(os.path.abspath(__file__)), filename)
    spec = importlib.util.spec_from_file_location(modname, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

def get_chroma_client():
    return chromadb.PersistentClient(path=CHROMA_DIR)

def build_store():
    documents_mod = _load_module("01_documents.py", "documents_mod")
    preprocessing_mod = _load_module("02_preprocessing.py", "preprocessing_mod")
    chunking_mod = _load_module("03_chunking.py", "chunking_mod")
    vector_mod = _load_module("04_vector_representation.py", "vector_mod")

    docs = documents_mod.load_documents()
    cleaned_docs = preprocessing_mod.preprocess_documents(docs)
    chunks = chunking_mod.chunk_documents(cleaned_docs)

    texts = [c["text"] for c in chunks]
    ids = [c["id"] for c in chunks]
    metadatas = [
        {"source": c["source"], "article_number": c["article_number"]}
        for c in chunks
    ]

    print(f"بنعمل embeddings لـ {len(texts)} مادة ...")
    embeddings = vector_mod.embed_texts(texts)

    client = get_chroma_client()
    
    try:
        client.delete_collection(COLLECTION_NAME)
    except Exception:
        pass
    collection = client.create_collection(COLLECTION_NAME)

    collection.add(ids=ids, documents=texts, metadatas=metadatas, embeddings=embeddings)

    print(f"تم تخزين {len(chunks)} مادة في ChromaDB بمجلد: {CHROMA_DIR}")
    return collection

if __name__ == "__main__":
    build_store()


In [ ]:
import os
import re
import importlib.util
import chromadb

CHROMA_DIR = os.path.join(os.path.dirname(os.path.abspath(__file__)), "chroma_db")
COLLECTION_NAME = "family_law"

STOPWORDS = {
    , "من", "في", "على", "الى", "إلى", "أن", "ان", "التي", "الذي",
    , "أو", "او", "ما", "لا", "لم", "يحق", "حق", "بشكل", "عن",
}

def _load_module(filename, modname):
    path = os.path.join(os.path.dirname(os.path.abspath(__file__)), filename)
    spec = importlib.util.spec_from_file_location(modname, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

def get_collection():
    client = chromadb.PersistentClient(path=CHROMA_DIR)
    return client.get_collection(COLLECTION_NAME)

def _keyword_score(query: str, text: str) -> int:
    
    words = [w for w in re.findall(r"[\u0621-\u064A]+", query) if w not in STOPWORDS and len(w) > 2]
    score = 0
    for w in words:
        if w in text:
            score += 1
    return score

def retrieve_context(query: str, top_k: int = 3):
    vector_mod = _load_module("04_vector_representation.py", "vector_mod")
    query_embedding = vector_mod.embed_texts([query])[0]

    collection = get_collection()
    total = collection.count()

    results = collection.query(query_embeddings=[query_embedding], n_results=total)

    docs = results.get("documents", [[]])[0]
    metas = results.get("metadatas", [[]])[0]
    dists = results.get("distances", [[]])[0]

    candidates = []
    max_dist = max(dists) if dists else 1.0
    for text, meta, dist in zip(docs, metas, dists):
        
        semantic_score = 1 - (dist / max_dist if max_dist else 0)
        keyword_score = _keyword_score(query, text)
        
        combined_score = semantic_score + (keyword_score * 0.5)
        candidates.append({
            : text,
            : meta.get("source"),
            : meta.get("article_number"),
            : dist,
            : keyword_score,
            : combined_score,
        })

    candidates.sort(key=lambda c: c["combined_score"], reverse=True)
    return candidates[:top_k]

if __name__ == "__main__":
    results = retrieve_context("هل يحق للزوجة طلب الخلع؟")
    for r in results:
        print(
            f"[{r['source']} - مادة {r['article_number']}] "
            f"(distance={r['distance']:.3f}, keyword_score={r['keyword_score']}, "
            f"combined={r['combined_score']:.3f})"
        )
        print(r["text"][:150])
        print()


In [ ]:
import os
from openai import OpenAI

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
OPENROUTER_MODEL = os.environ.get("OPENROUTER_MODEL", "openai/gpt-4o-mini")

SYSTEM_PROMPT = (
    
)

def build_prompt(query: str, context_chunks):
    context_text = "\n\n".join(
        f"[مصدر: {c['source']} - مادة {c['article_number']}]\n{c['text']}"
        for c in context_chunks
    )
    user_prompt = (
        f"السياق (مواد قانونية مسترجعة):\n{context_text}\n\n"
        f"سؤال المستخدم: {query}\n\n"
        
    )
    return user_prompt

def generate_answer(query: str, context_chunks):
    if not OPENROUTER_API_KEY:
        return (
            
        )

    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=OPENROUTER_API_KEY,
    )

    user_prompt = build_prompt(query, context_chunks)

    response = client.chat.completions.create(
        model=OPENROUTER_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
    )
    return response.choices[0].message.content

if __name__ == "__main__":
    import importlib.util

    def _load_module(filename, modname):
        path = os.path.join(os.path.dirname(os.path.abspath(__file__)), filename)
        spec = importlib.util.spec_from_file_location(modname, path)
        mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)
        return mod

    retrieve_mod = _load_module("06_retrieve_context.py", "retrieve_mod")
    q = "هل يحق للزوجة طلب الخلع؟"
    context = retrieve_mod.retrieve_context(q)
    print(generate_answer(q, context))


In [ ]:
import os
import importlib.util
import streamlit as st

BASE_DIR = os.path.dirname(os.path.abspath(__file__))

def load_module(filename, modname):
    path = os.path.join(BASE_DIR, filename)
    spec = importlib.util.spec_from_file_location(modname, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

retrieve_mod = load_module("06_retrieve_context.py", "retrieve_mod")
prompting_mod = load_module("07_prompting.py", "prompting_mod")

try:
    if not prompting_mod.OPENROUTER_API_KEY:
        prompting_mod.OPENROUTER_API_KEY = st.secrets.get("OPENROUTER_API_KEY", "")
    prompting_mod.OPENROUTER_MODEL = st.secrets.get(
        , prompting_mod.OPENROUTER_MODEL
    )
except Exception:
    pass

st.set_page_config(page_title="مساعد قانون الأسرة", page_icon="⚖️")
st.title("⚖️ مساعد قانون الأسرة المصري (RAG)")
st.caption(
    
)

if "history" not in st.session_state:
    st.session_state.history = []

query = st.text_input(
    ,
    placeholder="مثال: هل يحق للزوجة طلب الخلع؟",
)

top_k = st.sidebar.slider("عدد المواد المسترجعة (top_k)", 1, 5, 3)

if st.button("إسأل") and query.strip():
    with st.spinner("بندور على المواد القانونية المتعلقة بسؤالك..."):
        context_chunks = retrieve_mod.retrieve_context(query, top_k=top_k)

    with st.spinner("بنصيغ الإجابة..."):
        answer = prompting_mod.generate_answer(query, context_chunks)

    st.session_state.history.append(
        {"query": query, "answer": answer, "context": context_chunks}
    )

for item in reversed(st.session_state.history):
    st.markdown(f"### ❓ {item['query']}")
    st.markdown(item["answer"])
    with st.expander("📚 المواد القانونية المستخدمة (المصدر)"):
        for c in item["context"]:
            st.markdown(f"**{c['source']} - مادة {c['article_number']}**")
            st.write(c["text"])
            st.divider()
